In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/clean_orders.csv', parse_dates=['order_date', 'ship_date', 'order_month'])
d = df[(df['order_month'] >= '2015-01-01') & (df['order_month'] <= '2017-09-01')]
months = pd.date_range('2015-01-01', '2017-09-01', freq='MS')

A_cats = ['Fishing', 'Cleats', 'Camping & Hiking', 'Cardio Equipment',
          "Women's Apparel", 'Water Sports', 'Indoor/Outdoor Games']

dA = d[d['category'].isin(A_cats)]
series = dA.groupby(['category', 'order_month'])['qty'].sum().unstack(0).reindex(months, fill_value=0)

s = dA.groupby('category')[['sales', 'qty']].sum()
price = (s['sales'] / s['qty']).rename('price')
print(series.shape)
print(price.round(2))

(33, 7)
category
Camping & Hiking        299.98
Cardio Equipment         98.32
Cleats                   60.11
Fishing                 399.98
Indoor/Outdoor Games     49.98
Water Sports            200.39
Women's Apparel          50.00
Name: price, dtype: float64


In [2]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
import warnings
warnings.filterwarnings('ignore')

rows = []
for c in A_cats:
    fit = SimpleExpSmoothing(series[c], initialization_method='estimated').fit()
    resid = series[c] - fit.fittedvalues
    rows.append({'category': c,
                 'forecast_month': fit.forecast(1).iloc[0],
                 'sigma_month': resid.std()})
fc = pd.DataFrame(rows).set_index('category')
fc.round(1)

,forecast_month,sigma_month
category,,
Fishing,502.4,23.0
Cleats,2136.1,117.1
Camping & Hiking,398.2,17.2
Cardio Equipment,1087.1,88.9
Women's Apparel,1822.8,103.9
Water Sports,450.2,26.9
Indoor/Outdoor Games,1674.7,103.6


In [3]:
lt = dA.groupby('category')['days_real'].agg(lt_mean='mean', lt_std='std')
lt.round(2)

,lt_mean,lt_std
category,,
Camping & Hiking,3.50,1.62
Cardio Equipment,3.49,1.62
Cleats,3.51,1.63
Fishing,3.51,1.61
Indoor/Outdoor Games,3.50,1.63
Water Sports,3.48,1.62
Women's Apparel,3.49,1.62


In [4]:
from scipy.stats import norm

Z = norm.ppf(0.95)          # 1.645 for a 95% service level
COST_PCT, HOLD, ORDER_COST = 0.70, 0.20, 50

pol = fc.join(lt).join(price)
pol['d_day'] = pol['forecast_month'] / 30
pol['sigma_day'] = pol['sigma_month'] / np.sqrt(30)

# Safety stock with both demand and lead-time variability
pol['safety_stock'] = Z * np.sqrt(pol['lt_mean'] * pol['sigma_day']**2
                                  + pol['d_day']**2 * pol['lt_std']**2)
pol['reorder_point'] = pol['d_day'] * pol['lt_mean'] + pol['safety_stock']

pol['unit_cost'] = pol['price'] * COST_PCT
pol['EOQ'] = np.sqrt(2 * pol['forecast_month'] * 12 * ORDER_COST / (HOLD * pol['unit_cost']))
pol['safety_stock_value'] = pol['safety_stock'] * pol['unit_cost']

print(pol[['forecast_month', 'lt_mean', 'lt_std', 'safety_stock',
           'reorder_point', 'EOQ', 'safety_stock_value']].round(1))
print("\nTotal safety stock value: $", round(pol['safety_stock_value'].sum()))

                      forecast_month  lt_mean  lt_std  safety_stock  \
category                                                              
Fishing                        502.4      3.5     1.6          46.3   
Cleats                        2136.1      3.5     1.6         202.2   
Camping & Hiking               398.2      3.5     1.6          36.8   
Cardio Equipment              1087.1      3.5     1.6         108.7   
Women's Apparel               1822.8      3.5     1.6         172.1   
Water Sports                   450.2      3.5     1.6          42.8   
Indoor/Outdoor Games          1674.7      3.5     1.6         160.3   

                      reorder_point    EOQ  safety_stock_value  
category                                                        
Fishing                       105.1  103.8             12970.3  
Cleats                        451.8  551.9              8508.7  
Camping & Hiking               83.2  106.7              7718.7  
Cardio Equipment              235.2

In [5]:
var_demand = pol['lt_mean'] * pol['sigma_day']**2
var_lt = pol['d_day']**2 * pol['lt_std']**2
pol['lt_share_%'] = var_lt / (var_demand + var_lt) * 100
print(pol['lt_share_%'].round(0))

category
Fishing                 92.0
Cleats                  89.0
Camping & Hiking        93.0
Cardio Equipment        79.0
Women's Apparel         89.0
Water Sports            88.0
Indoor/Outdoor Games    87.0
Name: lt_share_%, dtype: float64


In [6]:
def ss_value(lt_f=1.0, dem_f=1.0):
    ss = Z * np.sqrt(pol['lt_mean'] * (pol['sigma_day'] * dem_f)**2
                     + pol['d_day']**2 * (pol['lt_std'] * lt_f)**2)
    return (ss * pol['unit_cost']).sum()

base = ss_value()
rows = []
for label, lt_f, dem_f in [('Current', 1, 1),
                           ('Lead-time variability -50%', 0.5, 1),
                           ('Forecast error -50%', 1, 0.5),
                           ('Both -50%', 0.5, 0.5)]:
    v = ss_value(lt_f, dem_f)
    rows.append({'scenario': label, 'safety_stock_$': round(v),
                 'reduction_%': round((1 - v / base) * 100, 1),
                 'annual_holding_saving_$': round((base - v) * HOLD)})
print(pd.DataFrame(rows).set_index('scenario'))

                            safety_stock_$  reduction_%  \
scenario                                                  
Current                              54308          0.0   
Lead-time variability -50%           31423         42.1   
Forecast error -50%                  51924          4.4   
Both -50%                            27154         50.0   

                            annual_holding_saving_$  
scenario                                             
Current                                           0  
Lead-time variability -50%                     4577  
Forecast error -50%                             477  
Both -50%                                      5431  


In [7]:
rng = np.random.default_rng(42)
DAYS = 360   # simulate 12 months

def simulate(c, ss_mult, lt_f=1.0, runs=200):
    p = pol.loc[c]
    mu_d, sd_d = p['d_day'], p['sigma_day']
    rop = mu_d * p['lt_mean'] + p['safety_stock'] * ss_mult
    q = p['EOQ']
    stockout_days, avg_inv = [], []
    for _ in range(runs):
        inv, pipeline, so, inv_sum = rop + q, [], 0, 0
        for day in range(DAYS):
            pipeline = [(t - 1, qty) for t, qty in pipeline]
            inv += sum(qty for t, qty in pipeline if t <= 0)
            pipeline = [(t, qty) for t, qty in pipeline if t > 0]
            dem = max(0, rng.normal(mu_d, sd_d))
            if dem > inv:
                so += 1
            inv = max(0, inv - dem)
            on_order = sum(qty for _, qty in pipeline)
            if inv + on_order <= rop:
                lt = max(1, round(rng.normal(p['lt_mean'], p['lt_std'] * lt_f)))
                pipeline.append((lt, q))
            inv_sum += inv
        stockout_days.append(so / DAYS * 100)
        avg_inv.append(inv_sum / DAYS)
    return np.mean(stockout_days), np.mean(avg_inv) * p['unit_cost']

rows = []
for label, ss_mult, lt_f in [('No safety stock', 0.0, 1.0),
                             ('Current policy (95% SS)', 1.0, 1.0),
                             ('Proposed: lead-time variability -50%', 1.0, 0.5)]:
    res_ = [simulate(c, ss_mult, lt_f) for c in A_cats]
    rows.append({'policy': label,
                 'avg_stockout_days_%': round(np.mean([r[0] for r in res_]), 2),
                 'avg_inventory_$': round(sum(r[1] for r in res_))})
print(pd.DataFrame(rows).set_index('policy'))

                                      avg_stockout_days_%  avg_inventory_$
policy                                                                    
No safety stock                                      7.76            81819
Current policy (95% SS)                              0.54           131830
Proposed: lead-time variability -50%                 0.02           132243


In [8]:
rows = []
for ss_mult in [1.0, 0.8, 0.6, 0.5, 0.4]:
    res_ = [simulate(c, ss_mult, lt_f=0.5) for c in A_cats]
    rows.append({'safety_stock_vs_current': f'{int(ss_mult*100)}%',
                 'avg_stockout_days_%': round(np.mean([r[0] for r in res_]), 2),
                 'avg_inventory_$': round(sum(r[1] for r in res_))})
print(pd.DataFrame(rows).set_index('safety_stock_vs_current'))

                         avg_stockout_days_%  avg_inventory_$
safety_stock_vs_current                                      
100%                                    0.03           132270
80%                                     0.10           121519
60%                                     0.35           110488
50%                                     0.58           105244
40%                                     0.95            99919


In [9]:
pol.round(2).to_csv('../data/inventory_policy.csv')
sim = pd.DataFrame(rows)
sim.to_csv('../data/safety_stock_scenarios.csv', index=False)